In [22]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn
import joblib

base_path = r"C:\BSAI\3rd Semester\PROGRAMING FOR AI\LAB\week9"

os.makedirs(os.path.join(base_path, "data"), exist_ok=True)
os.makedirs(os.path.join(base_path, "models"), exist_ok=True)
os.makedirs(os.path.join(base_path, "reports"), exist_ok=True)

DATA_PATH = Path(base_path) / "data" / "machines_ml.csv"
machines = pd.read_csv(DATA_PATH)

required = {"temp_c", "vibration", "runtime", "pressure", "fault"}
missing = required - set(machines.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
if machines[list(required)].isna().any().any():
    raise ValueError("Required columns contain missing values.")
if not set(machines["fault"].unique()).issubset({0, 1}):
    raise ValueError("fault must contain only 0 and 1.")
if machines["fault"].nunique() != 2:
    raise ValueError("Both Normal (0) and Fault (1) classes must be present.")

feature_cols = ["temp_c", "vibration", "runtime", "pressure"]

print(f"Rows: {machines.shape[0]}, Columns: {machines.shape[1]}")

print("\nDescriptive statistics:")
display(machines[list(required)].describe().round(2))

print("\nClass counts (0=Normal; 1=Fault):")
print(machines["fault"].value_counts().sort_index())

print("\nTarget class proportions:")
display(
    machines["fault"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .rename(index={0: "Normal (0)", 1: "Fault (1)"})
    .rename("Proportion (%)")
    .round(2)
)

Rows: 160, Columns: 6

Descriptive statistics:


,fault,pressure,vibration,runtime,temp_c
count,160.00,160.00,160.00,160.00,160.00
mean,0.30,5.22,3.56,10150.25,76.24
std,0.46,1.28,1.48,5317.59,15.32
min,0.00,2.00,0.60,524.00,43.00
25%,0.00,4.40,2.40,5409.75,65.75
50%,0.00,5.20,3.70,10659.50,76.50
75%,1.00,6.02,4.50,14581.75,87.00
max,1.00,8.60,7.60,18506.00,116.00



Class counts (0=Normal; 1=Fault):
fault
0    112
1     48
Name: count, dtype: int64

Target class proportions:


fault
Normal (0)    70.0
Fault (1)     30.0
Name: Proportion (%), dtype: float64

In [23]:
from sklearn.model_selection import train_test_split

FEATURES = ["temp_c", "vibration", "runtime", "pressure"]
X = machines[FEATURES]
y = machines["fault"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Train:", X_train.shape, y_train.value_counts().to_dict())
print("Test:", X_test.shape, y_test.value_counts().to_dict())

distribution = pd.DataFrame({
    "Original count": y.value_counts().sort_index(),
    "Original %": y.value_counts(normalize=True).sort_index() * 100,
    "Training count": y_train.value_counts().sort_index(),
    "Training %": y_train.value_counts(normalize=True).sort_index() * 100,
    "Testing count": y_test.value_counts().sort_index(),
    "Testing %": y_test.value_counts(normalize=True).sort_index() * 100,
})
print("\nData Split Distribution:")
display(distribution.round(2))
X_train_again, X_test_again, _, _ = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
assert X_train.equals(X_train_again) and X_test.equals(X_test_again)
print("\nRepeated split with random_state=42 is identical.")

Train: (120, 4) {0: 84, 1: 36}
Test: (40, 4) {0: 28, 1: 12}

Data Split Distribution:


,Original count,Original %,Training count,Training %,Testing count,Testing %
fault,,,,,,
0,112,70.0,84,70.0,28,70.0
1,48,30.0,36,30.0,12,30.0



Repeated split with random_state=42 is identical.


In [24]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

test_predictions = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

cm = confusion_matrix(y_test, test_predictions)
tn, fp, fn, tp = cm.ravel()

manual_accuracy = (tn + tp) / cm.sum()
print({"TN": tn, "FP": fp, "FN": fn, "TP": tp})
print(f"Test Accuracy: {test_accuracy}")

assert abs(manual_accuracy - test_accuracy) < 1e-12
print("Manual accuracy verification successful!")

{'TN': np.int64(25), 'FP': np.int64(3), 'FN': np.int64(5), 'TP': np.int64(7)}
Test Accuracy: 0.8
Manual accuracy verification successful!


In [25]:
import joblib

results = []
for depth in [1, 2, 3, 5, None]:
    candidate = DecisionTreeClassifier(max_depth=depth, random_state=42)
    candidate.fit(X_train, y_train)
    
    train_score = candidate.score(X_train, y_train)
    test_score = candidate.score(X_test, y_test)
    
    results.append({
        "max_depth": str(depth),
        "train_accuracy": train_score,
        "test_accuracy": test_score,
        "gap": train_score - test_score
    })

comparison = pd.DataFrame(results)
comparison_path = Path(base_path) / "reports" / "model_comparison.csv"
comparison.to_csv(comparison_path, index=False)
print("Model Comparison Saved:\n", comparison)

bundle_path = Path(base_path) / "models" / "machine_predictor.joblib"
bundle = {"model": model, "features": FEATURES}
joblib.dump(bundle, bundle_path)

loaded = joblib.load(bundle_path)
reloaded_predictions = loaded["model"].predict(X_test[loaded["features"]])

assert (reloaded_predictions == test_predictions).all()
print("Model reloading verified successfully!")

Model Comparison Saved:
   max_depth  train_accuracy  test_accuracy       gap
0         1        0.808333          0.800  0.008333
1         2        0.808333          0.800  0.008333
2         3        0.891667          0.800  0.091667
3         5        0.991667          0.825  0.166667
4      None        1.000000          0.825  0.175000
Model reloading verified successfully!


In [26]:
new_machines = pd.DataFrame([
    {"temp_c": 98, "vibration": 6.5, "runtime": 14200, "pressure": 3.2},
    {"temp_c": 61, "vibration": 1.2, "runtime": 2700, "pressure": 6.7},
    {"temp_c": 81, "vibration": 3.8, "runtime": 8600, "pressure": 4.8},
])

feature_order = loaded["features"]
if set(new_machines.columns) != set(feature_order):
    raise ValueError("Prediction columns do not match training features.")

new_machines["predicted_fault"] = loaded["model"].predict(new_machines[feature_order])

print("New Predictions Table:")
print(new_machines)

New Predictions Table:
   temp_c  vibration  runtime  pressure  predicted_fault
0      98        6.5    14200       3.2                1
1      61        1.2     2700       6.7                0
2      81        3.8     8600       4.8                0
